# KuwaPIPE — PS 侧控制脚本（PYNQ）

本 Notebook 负责四件事：

1. 加载 Overlay（`base.bit` + `base.hwh`）；
2. 通过 I2C 初始化 OV5640，输出 **800x480 RGB565**；
3. 配置 `VideoMode(640, 480, 24)`，建立「摄像头 -> PL 处理链 -> HDMI」直通；
4. 抓帧保存 + 对硬件描述做自检（确认跑的是自研模块）。

硬件：xc7z020clg400-2（PYNQ-Z2 兼容板）+ OV5640 + HDMI 显示器。

> **运行环境**：需使用开发板厂商的 PYNQ 镜像（`Release 2025_08_27`）。
> `pynq.lib.ov5640` 是该镜像添加的（官方 PYNQ 无此模块），`BaseOverlay` 也被该镜像改过；
> 官方镜像上运行会在 `import OV5640` 处报 ImportError。详见 `PS_CODE_NOTES.md` 第 6 节。

> **显示分辨率只能用 640x480**：PYNQ 的 DVI 输出只有
> `(640,480) (800,600) (1280,720) (1280,1024) (1920,1080)` 五种模式，
> 上游链路的 800x480 不在其中，写 `VideoMode(800, 480, 24)` 会在 `hdmi.configure()`
> 抛 `ValueError: Invalid Output resolution 800x480`。
> 因此每行只显示前 640 像素，**右侧 20% 被裁掉** —— 这是显示通路的硬限制，不是代码问题。

In [ ]:
import os
from datetime import datetime

import numpy as np
import PIL.Image

from pynq.overlays.base import BaseOverlay
from pynq.lib.iic import AxiIIC
from pynq.lib.ov5640 import OV5640
from pynq.lib.video import PIXEL_RGB, VideoMode

# ---------------- 配置 ----------------
BIT_PATH    = "./base.bit"
HWH_PATH    = "./base.hwh"
FRAME_W     = 640          # 显示模式受 PYNQ DVI 限制（800x480 不支持），右侧会裁 20%
FRAME_H     = 480
FRAME_BPP   = 24
CAPTURE_DIR = "./captures"
os.makedirs(CAPTURE_DIR, exist_ok=True)

# ---------------- 加载 Overlay ----------------
base = BaseOverlay(BIT_PATH)
cam  = base.video.camera_in
hdmi = base.video.hdmi_out
print("overlay loaded:", BIT_PATH)

In [ ]:
# ---------------- OV5640 初始化（I2C）----------------
iic = AxiIIC(base.ip_dict["cam_iic"])
sensor = OV5640(0x3C, iic)     # OV5640 7-bit I2C 地址
sensor.init()                  # 寄存器表固定输出 800x480 RGB565
print("OV5640 initialised (800x480)")

In [ ]:
# ---------------- 视频通路 ----------------
mode = VideoMode(FRAME_W, FRAME_H, FRAME_BPP)
cam.configure(mode, PIXEL_RGB)
hdmi.configure(cam.mode, PIXEL_RGB)
cam.start()
hdmi.start()
cam.tie(hdmi)                  # 直通显示；PL 链路内已含 中值 -> Kuwahara -> 风格化
print("video running:", cam.mode)

## 抓一帧保存

`camera_in.readframe()` 返回的是 PL 送进 VDMA 的帧；直通模式下它同时出现在 HDMI 上。

In [ ]:
# ---------------- 抓一帧并保存 ----------------
frame = cam.readframe()        # numpy (H, W, 3), uint8
img = PIL.Image.fromarray(frame)
out = os.path.join(CAPTURE_DIR,
                   "frame_%s.png" % datetime.now().strftime("%Y%m%d_%H%M%S"))
img.save(out)
print("saved:", out, frame.shape, frame.dtype)
img

In [ ]:
# ---------------- 硬件自检：确认跑的是自研模块 ----------------
with open(HWH_PATH, "r", encoding="utf-8", errors="ignore") as f:
    hwh = f.read()

checks = {
    "ov5640_capture (自研采集 IP)":  "ov5640_capture" in hwh,
    "video_kuwahara (Kuwahara)":     "video_kuwahara" in hwh,
    "video_mode_switch (按键切换)":   "video_mode_switch" in hwh,
    "video_style_bypass (风格化旁路)": "video_style_bypass" in hwh,
}
for k, v in checks.items():
    print("%-34s %s" % (k, "OK" if v else "MISSING"))

In [ ]:
# ---------------- 按键状态（切换由 PL 完成，这里只读）----------------
try:
    raw = base.buttons.read()
    print("buttons raw =", bin(raw))
    print("提示：PL_KEY0 由 PL 侧消抖并切换 处理/直通，LED2 亮 = 直通")
except Exception as e:
    print("buttons unavailable:", e)

## 收尾

停止前建议先执行下面的单元，避免 DMA 占用。

In [ ]:
# ---------------- 收尾 ----------------
hdmi.stop()
cam.stop()
print("video stopped")